# 🍎 Obsterkennung – Bildklassifikation im Browser
### MobileNet · TensorFlow.js · On-Device KI

Dieses Notebook zeigt **Bildklassifikation** mit einem vortrainierten neuronalen Netz.  
Das Modell **MobileNetV2** wurde auf **ImageNet** trainiert – einem Datensatz mit  
über 14 Millionen Bildern in 1000 Kategorien, darunter viele Obstsorten.

---
**Lernziele**
- Konzept der Bildklassifikation verstehen
- Vortrainierte Modelle (Transfer Learning) einsetzen
- Wahrscheinlichkeits-Output von KI-Modellen interpretieren


## 1 · Hintergrund: Wie funktioniert Bildklassifikation?

```
Foto (Pixel)  →  [MobileNetV2]  →  1000 Wahrscheinlichkeiten
                      ↑
        Trainiert auf ImageNet (14 Mio. Bilder)
```

MobileNetV2 gibt für **jede der 1000 Klassen** eine Wahrscheinlichkeit aus.  
Wir filtern diese Liste auf bekannte Obstsorten und zeigen das beste Ergebnis.

**Warum MobileNet?**  
MobileNet wurde speziell für mobile Geräte entwickelt – klein, schnell, effizient.  
Es läuft problemlos im Browser, ohne GPU-Server.


## 2 · Die App ausführen

Foto-Button drücken → Kamerabild wird eingefroren → MobileNet analysiert → Ergebnis erscheint.


In [ ]:
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8"/>
<style>
  :root {
    --bg: #0f1a0a; --surface: #172110; --border: #2a3d1e;
    --accent: #7eff4f; --accent2: #ffdd00;
    --text: #dff0d0; --muted: #4a6638; --radius: 12px;
    --font: "Courier New", monospace;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: var(--bg); color: var(--text);
    font-family: var(--font); padding: 16px;
    display: flex; flex-direction: column; align-items: center; gap: 12px;
  }
  h2 { font-size: .95rem; letter-spacing: .18em; color: var(--accent); text-transform: uppercase; }
  #status {
    font-size: .68rem; letter-spacing: .14em; padding: 4px 14px;
    border: 1px solid var(--border); border-radius: 999px; color: var(--muted); transition: all .3s;
  }
  #status.ready   { color: var(--accent); border-color: var(--accent); box-shadow: 0 0 10px #7eff4f30; }
  #status.loading { color: var(--accent2); border-color: var(--accent2); animation: blink 1s infinite; }
  #status.error   { color: #ff4444; border-color: #ff4444; }
  @keyframes blink { 50%{opacity:.4;} }
  #wrap {
    position: relative; width: 480px; max-width: 100%; aspect-ratio: 4/3;
    background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden;
  }
  #video  { display: none; }
  #canvas { position: absolute; inset: 0; width: 100%; height: 100%; display: block; }
  #scanline {
    position: absolute; left: 0; right: 0; height: 2px;
    background: linear-gradient(90deg,transparent,var(--accent),transparent);
    box-shadow: 0 0 12px var(--accent); top: 0; display: none;
    animation: scan 1.2s linear infinite;
  }
  @keyframes scan { from{top:0;} to{top:100%;} }
  #cross { position:absolute; inset:0; display:flex; align-items:center; justify-content:center; pointer-events:none; }
  #cross::before,#cross::after { content:''; position:absolute; background:rgba(126,255,79,.3); }
  #cross::before { width:1px; height:50px; }
  #cross::after  { width:50px; height:1px; }
  #placeholder {
    position: absolute; inset: 0; display: flex; align-items: center; justify-content: center;
    color: var(--muted); font-size: .75rem; letter-spacing: .1em;
  }
  #btn-flip {
    position: absolute; top: 10px; right: 10px; z-index: 10;
    width: 36px; height: 36px; border-radius: 50%;
    background: rgba(0,0,0,.5); border: 1px solid rgba(126,255,79,.4);
    color: var(--accent); font-size: 1.2rem; cursor: pointer;
    display: none; align-items: center; justify-content: center;
    transition: background .2s, box-shadow .2s;
  }
  #btn-flip:hover { background: rgba(126,255,79,.2); box-shadow: 0 0 10px #7eff4f40; }
  #cam-label {
    position: absolute; top: 10px; left: 10px; z-index: 10;
    font-size: .55rem; letter-spacing: .12em; color: rgba(126,255,79,.8);
    background: rgba(0,0,0,.45); padding: 3px 8px; border-radius: 999px; display: none;
  }
  #btn-shoot {
    width:64px; height:64px; border-radius:50%;
    border:3px solid var(--accent); background:transparent; cursor:pointer;
    position:relative; transition:transform .1s, box-shadow .2s;
  }
  #btn-shoot::after {
    content:''; position:absolute; inset:8px; border-radius:50%;
    background:var(--accent); transition:background .2s;
  }
  #btn-shoot:active  { transform:scale(.92); }
  #btn-shoot:hover   { box-shadow:0 0 18px #7eff4f50; }
  #btn-shoot:disabled::after { background:var(--muted); }
  #btn-shoot:disabled { border-color:var(--muted); cursor:default; }
  #result {
    width:480px; max-width:100%; background:var(--surface);
    border:1px solid var(--border); border-radius:var(--radius);
    padding:22px 18px; text-align:center; display:none; animation:up .4s ease;
  }
  @keyframes up { from{opacity:0;transform:translateY(14px);} to{opacity:1;transform:none;} }
  #emoji    { font-size:4rem; margin-bottom:8px; filter:drop-shadow(0 0 14px #7eff4f50); }
  #label    { font-size:1.5rem; letter-spacing:.1em; color:var(--accent); text-transform:uppercase; margin-bottom:4px; }
  #label-de { font-size:.85rem; color:var(--text); margin-bottom:14px; }
  .bar-wrap { width:75%; margin:0 auto; }
  .bar-lbl  { font-size:.6rem; letter-spacing:.12em; color:var(--muted); margin-bottom:4px; }
  .bar-bg   { height:5px; background:var(--border); border-radius:999px; overflow:hidden; }
  .bar-fill { height:100%; border-radius:999px; background:var(--accent); box-shadow:0 0 8px var(--accent); width:0%; transition:width .6s ease; }
  #pct      { font-size:.72rem; color:var(--accent2); margin-top:4px; }
  #no-fruit { font-size:.75rem; color:var(--muted); margin-top:8px; display:none; }
  .btn-row  { display:flex; gap:10px; justify-content:center; margin-top:16px; flex-wrap:wrap; }
  #btn-retry, #btn-send {
    padding:9px 20px; background:transparent;
    font-family:var(--font); font-size:.68rem; letter-spacing:.14em;
    text-transform:uppercase; border-radius:var(--radius); cursor:pointer;
    display:none; transition:border-color .2s,color .2s;
  }
  #btn-retry { border:1px solid var(--border); color:var(--muted); }
  #btn-retry:hover { border-color:var(--accent); color:var(--accent); }
  #btn-send  { border:1px solid #ffdd0066; color:var(--accent2); }
  #btn-send:hover { border-color:var(--accent2); box-shadow:0 0 10px #ffdd0030; }
  #send-status { font-size:.62rem; letter-spacing:.1em; color:var(--muted); margin-top:6px; min-height:1em; }
</style>
</head>
<body>
<h2>Obsterkennung · MobileNet</h2>
<div id="status">LADE MODELL …</div>
<div id="wrap">
  <video id="video" autoplay playsinline muted></video>
  <canvas id="canvas"></canvas>
  <div id="scanline"></div>
  <div id="cross"></div>
  <div id="placeholder">▶ WIRD GESTARTET …</div>
  <button id="btn-flip">⟳</button>
  <div id="cam-label">–</div>
</div>
<button id="btn-shoot" disabled></button>
<div id="result">
  <div id="emoji">🍎</div>
  <div id="label">–</div>
  <div id="label-de"></div>
  <div class="bar-wrap">
    <div class="bar-lbl">SICHERHEIT</div>
    <div class="bar-bg"><div class="bar-fill" id="bar"></div></div>
    <div id="pct">–</div>
  </div>
  <div id="no-fruit">Kein Obst erkannt – versuch es nochmal!</div>
  <div class="btn-row">
    <button id="btn-retry">↩ NOCHMAL</button>
    <button id="btn-send">📤 AN PYTHON SENDEN</button>
  </div>
  <div id="send-status"></div>
</div>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.17.0/dist/tf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@tensorflow-models/mobilenet@2.1.1/dist/mobilenet.min.js"></script>
<script>
const video    = document.getElementById('video');
const canvas   = document.getElementById('canvas');
const ctx      = canvas.getContext('2d');
const scanline = document.getElementById('scanline');
const btnShoot = document.getElementById('btn-shoot');
const btnRetry = document.getElementById('btn-retry');
const btnFlip  = document.getElementById('btn-flip');
const btnSend  = document.getElementById('btn-send');
const camLabel = document.getElementById('cam-label');
const statusEl = document.getElementById('status');
const resultDiv= document.getElementById('result');
const bar      = document.getElementById('bar');
const ph       = document.getElementById('placeholder');
const sendSt   = document.getElementById('send-status');

const offscreen = document.createElement('canvas');
const offCtx    = offscreen.getContext('2d');

let model       = null;
let liveRunning = false;
let camDevices  = [];
let camIndex    = 0;
let mirrorFrame = false;

const FRUITS = [
  { keywords:['banana'],                         en:'Banana',       de:'Banane',         emoji:'🍌' },
  { keywords:['apple','granny smith','bramley'],  en:'Apple',        de:'Apfel',          emoji:'🍎' },
  { keywords:['orange','clementine','mandarin','tangerine','navel'], en:'Orange', de:'Orange', emoji:'🍊' },
  { keywords:['lemon','lime'],                    en:'Lemon/Lime',   de:'Zitrone/Limette',emoji:'🍋' },
  { keywords:['strawberry'],                      en:'Strawberry',   de:'Erdbeere',       emoji:'🍓' },
  { keywords:['pineapple'],                       en:'Pineapple',    de:'Ananas',         emoji:'🍍' },
  { keywords:['grape','grapevine'],               en:'Grapes',       de:'Weintrauben',    emoji:'🍇' },
  { keywords:['watermelon'],                      en:'Watermelon',   de:'Wassermelone',   emoji:'🍉' },
  { keywords:['mango'],                           en:'Mango',        de:'Mango',          emoji:'🥭' },
  { keywords:['peach','nectarine'],               en:'Peach',        de:'Pfirsich',       emoji:'🍑' },
  { keywords:['pear'],                            en:'Pear',         de:'Birne',          emoji:'🍐' },
  { keywords:['cherry','morello'],                en:'Cherry',       de:'Kirsche',        emoji:'🍒' },
  { keywords:['kiwi'],                            en:'Kiwi',         de:'Kiwi',           emoji:'🥝' },
  { keywords:['coconut'],                         en:'Coconut',      de:'Kokosnuss',      emoji:'🥥' },
  { keywords:['avocado'],                         en:'Avocado',      de:'Avocado',        emoji:'🥑' },
  { keywords:['blueberry'],                       en:'Blueberry',    de:'Blaubeere',      emoji:'🫐' },
  { keywords:['raspberry'],                       en:'Raspberry',    de:'Himbeere',       emoji:'🍓' },
  { keywords:['melon','honeydew','cantaloupe'],   en:'Melon',        de:'Melone',         emoji:'🍈' },
  { keywords:['plum','damson'],                   en:'Plum',         de:'Pflaume',        emoji:'🍑' },
  { keywords:['papaya','pawpaw'],                 en:'Papaya',       de:'Papaya',         emoji:'🍈' },
];

function findFruit(className) {
  const lc = className.toLowerCase();
  for (const f of FRUITS) {
    if (f.keywords.some(k => lc.includes(k))) return f;
  }
  return null;
}

function isFrontCamera(label) {
  const lc = (label || '').toLowerCase();
  return lc.includes('front') || lc.includes('user') || lc.includes('selfie') || lc.includes('facetime');
}

function updateCamLabel() {
  const dev = camDevices[camIndex];
  const lbl = dev ? dev.label : '';
  const track = video.srcObject && video.srcObject.getVideoTracks()[0];
  const settings = track ? track.getSettings() : {};
  if (camDevices.length <= 1) {
    camLabel.textContent = 'KAMERA';
  } else if (settings.facingMode === 'user' || isFrontCamera(lbl)) {
    camLabel.textContent = 'FRONT';
  } else if (settings.facingMode === 'environment') {
    camLabel.textContent = 'RÜCK';
  } else {
    camLabel.textContent = 'KAM ' + (camIndex + 1);
  }
  mirrorFrame = settings.facingMode === 'user' || isFrontCamera(lbl);
}

function drawLiveFrame() {
  if (!liveRunning) return;
  const w = canvas.width, h = canvas.height;
  ctx.save();
  if (mirrorFrame) { ctx.translate(w, 0); ctx.scale(-1, 1); }
  ctx.drawImage(video, 0, 0, w, h);
  ctx.restore();
  requestAnimationFrame(drawLiveFrame);
}

async function startStream(deviceId) {
  if (video.srcObject) {
    video.srcObject.getTracks().forEach(t => t.stop());
    video.srcObject = null;
  }
  const constraints = deviceId
    ? { video: { deviceId: { exact: deviceId } } }
    : { video: true };
  const stream = await navigator.mediaDevices.getUserMedia(constraints);
  video.srcObject = stream;
  await new Promise(r => video.onloadedmetadata = r);
  await video.play();
  await new Promise(r => setTimeout(r, 400));
  const w = video.videoWidth  || 640;
  const h = video.videoHeight || 480;
  canvas.width = offscreen.width  = w;
  canvas.height= offscreen.height = h;
  updateCamLabel();
}

async function loadDevices() {
  const all = await navigator.mediaDevices.enumerateDevices();
  camDevices = all.filter(d => d.kind === 'videoinput');
}

async function init() {
  try {
    setStatus('LADE MODELL …','loading');
    model = await mobilenet.load({ version:2, alpha:1.0 });
    setStatus('STARTE KAMERA …','loading');
    await startStream(null);
    await loadDevices();
    const backIdx = camDevices.findIndex(d => {
      const lc = (d.label||'').toLowerCase();
      return lc.includes('back') || lc.includes('rear') || lc.includes('environment');
    });
    if (backIdx >= 0 && backIdx !== 0) {
      camIndex = backIdx;
      await startStream(camDevices[camIndex].deviceId);
    } else {
      camIndex = 0;
    }
    ph.style.display      = 'none';
    camLabel.style.display= 'block';
    if (camDevices.length > 1) btnFlip.style.display = 'flex';
    liveRunning = true;
    drawLiveFrame();
    setStatus('BEREIT – FOTO AUFNEHMEN','ready');
    btnShoot.disabled = false;
  } catch(e) {
    setStatus('FEHLER: ' + e.message,'error');
  }
}

btnFlip.addEventListener('click', async () => {
  if (!liveRunning) return;
  const prevIndex = camIndex;
  camIndex = (camIndex + 1) % camDevices.length;
  try {
    await startStream(camDevices[camIndex].deviceId);
  } catch(e) {
    camIndex = prevIndex;
    setStatus('Kamera nicht verfügbar','error');
    setTimeout(() => setStatus('BEREIT – FOTO AUFNEHMEN','ready'), 2000);
  }
});

btnShoot.addEventListener('click', async () => {
  liveRunning = false;
  scanline.style.display = 'block';
  btnShoot.disabled = true;
  sendSt.textContent = '';
  setStatus('ANALYSIERE …','loading');

  offCtx.save();
  if (mirrorFrame) { offCtx.translate(offscreen.width, 0); offCtx.scale(-1, 1); }
  offCtx.drawImage(video, 0, 0, offscreen.width, offscreen.height);
  offCtx.restore();
  ctx.drawImage(offscreen, 0, 0);

  await new Promise(r => setTimeout(r, 80));
  const preds = await model.classify(offscreen, 5);
  scanline.style.display = 'none';

  let best = null;
  for (const p of preds) {
    const f = findFruit(p.className);
    if (f) { best = { ...f, prob: p.probability }; break; }
  }

  resultDiv.style.display = 'block';
  btnRetry.style.display  = 'inline-block';
  btnSend.style.display   = 'inline-block';

  if (best) {
    document.getElementById('no-fruit').style.display = 'none';
    document.getElementById('emoji').textContent    = best.emoji;
    document.getElementById('label').textContent    = best.en;
    document.getElementById('label-de').textContent = best.de;
    bar.style.width = Math.round(best.prob * 100) + '%';
    document.getElementById('pct').textContent = Math.round(best.prob * 100) + ' %';
    setStatus('ERKANNT: ' + best.de.toUpperCase(),'ready');
  } else {
    document.getElementById('no-fruit').style.display = 'block';
    document.getElementById('emoji').textContent    = '❓';
    document.getElementById('label').textContent    = preds[0]?.className?.split(',')[0] ?? '–';
    document.getElementById('label-de').textContent = 'Nicht in Obstliste';
    bar.style.width = Math.round((preds[0]?.probability ?? 0) * 100) + '%';
    document.getElementById('pct').textContent = Math.round((preds[0]?.probability ?? 0) * 100) + ' %';
    setStatus('KEIN OBST ERKANNT','');
  }
});

// Jupyter-Server-Basis-URL ermitteln (JupyterHub & JupyterLab)
function getApiBase() {
  const path = parent.location.pathname;
  // JupyterHub: /user/<name>/lab/...  →  /user/<name>/
  const hub = path.match(/^(\\/user\\/[^/]+\\/)/);
  if (hub) return parent.location.origin + hub[1];
  return parent.location.origin + '/';
}

// Bild über Jupyter Contents API als Datei speichern → Python liest sie
btnSend.addEventListener('click', async () => {
  try {
    sendSt.style.color = '#aaa';
    sendSt.textContent = 'Sende …';
    const b64    = offscreen.toDataURL('image/jpeg', 0.85).split(',')[1];
    const apiUrl = getApiBase() + 'api/contents/captured_fruit.txt';
    const xsrf   = parent.document.cookie.match(/_xsrf=([^;]+)/)?.[1] ?? '';
    const resp   = await fetch(apiUrl, {
      method : 'PUT',
      headers: { 'Content-Type': 'application/json', 'X-XSRFToken': xsrf },
      body   : JSON.stringify({ type: 'file', format: 'text', content: b64 })
    });
    if (!resp.ok) throw new Error('HTTP ' + resp.status);
    sendSt.style.color = '#7eff4f';
    sendSt.textContent = '✓ Bild gespeichert – Zelle 3 jetzt ausführen!';
  } catch(e) {
    sendSt.style.color = '#ff4444';
    sendSt.textContent = '✗ Fehler: ' + e.message;
  }
});

btnRetry.addEventListener('click', () => {
  resultDiv.style.display = 'none';
  btnRetry.style.display  = 'none';
  btnSend.style.display   = 'none';
  sendSt.textContent      = '';
  document.getElementById('no-fruit').style.display = 'none';
  bar.style.width = '0%';
  btnShoot.disabled = false;
  liveRunning = true;
  drawLiveFrame();
  setStatus('BEREIT – FOTO AUFNEHMEN','ready');
});

function setStatus(t,c){ statusEl.textContent=t; statusEl.className=c; }
init();
</script>
</body></html>
"""
HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;").replace(chr(10), "&#10;")}" width="560" height="720" allow="camera" style="border:none;border-radius:12px;"></iframe>')

## 3 · Was gibt das Modell zurück?

Lass uns die rohen Modell-Ausgaben direkt in Python betrachten – dafür laden wir das Modell serverseitig nach.


In [ ]:
import tensorflow as tf
import numpy as np
import requests
import base64
import os
from PIL import Image
from io import BytesIO

# ── Modell laden (nur beim ersten Aufruf) ─────────────────────────────────────
if 'mobilenet_model' not in dir() or mobilenet_model is None:
    print("Lade MobileNetV2 …")
    mobilenet_model = tf.keras.applications.MobileNetV2(weights='imagenet')
else:
    print("MobileNetV2 bereits geladen – überspringe.")

# ── Bildquelle ────────────────────────────────────────────────────────────────
# Wurde ein Kamerabild per "📤 AN PYTHON SENDEN" übertragen?
# Falls ja, wird captured_fruit.txt im Home-Verzeichnis gelesen.
fallback_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/800px-Red_Apple.jpg"

cam_file = os.path.expanduser("~/captured_fruit.txt")

if os.path.exists(cam_file):
    with open(cam_file) as f:
        b64 = f.read().strip()
    img_data = base64.b64decode(b64)
    img = Image.open(BytesIO(img_data)).convert("RGB").resize((224, 224))
    bildquelle = "📷 Kamerabild"
else:
    # Kein Kamerabild vorhanden → Beispielbild von URL verwenden
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Jupyter/1.0)"}
    response = requests.get(fallback_url, headers=headers, timeout=10)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB").resize((224, 224))
    bildquelle = "🌐 " + fallback_url.split('/')[-1]

# ── Vorverarbeitung ───────────────────────────────────────────────────────────
x = tf.keras.applications.mobilenet_v2.preprocess_input(
        np.array(img, dtype=np.float32)[np.newaxis])

# ── Vorhersage ────────────────────────────────────────────────────────────────
preds = mobilenet_model.predict(x, verbose=0)
top5  = tf.keras.applications.mobilenet_v2.decode_predictions(preds, top=5)[0]

# ── Ausgabe ───────────────────────────────────────────────────────────────────
print(f"\nBild: {bildquelle}")
print("Rohe Modell-Ausgabe (Top 5):")
print("-" * 45)
for i, (_, name, score) in enumerate(top5):
    bar = "█" * int(score * 40)
    print(f"{i+1}. {name:<25} {score*100:5.1f}%  {bar}")

total = preds[0][preds[0].argsort()[-5:][::-1]].sum()
print(f"\nSumme Top-5: {total:.4f}  (alle 1000 Klassen summieren sich auf 1.0)")

## 4 · Aufgaben zum Experimentieren

**Aufgabe 1 – Neue Obstsorte hinzufügen**  
Ergänze in der `FRUITS`-Liste eine neue Obstsorte.  
Recherchiere zuerst, wie ImageNet die Sorte nennt (englisch!).  
→ [ImageNet-Klassenliste (1000 Klassen)](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a)

**Aufgabe 2 – Konfidenzschwelle**  
Ändere den Code so, dass das Ergebnis nur angezeigt wird, wenn die Wahrscheinlichkeit über 50% liegt.  
```javascript
if (best && best.prob > 0.5) { ... }
```

**Aufgabe 3 – Top-3 anzeigen** *(Fortgeschritten)*  
Statt nur dem besten Ergebnis: Zeige die drei wahrscheinlichsten Obstsorten mit je einem Balken.

## 5 · Grenzen des Modells – Diskussion

MobileNet wurde auf **ImageNet** trainiert, nicht speziell auf Obst.  
Das hat Konsequenzen:

| Situation | Was passiert? |
|-----------|---------------|
| Exotisches Obst (Physalis, Lychee) | Wird nicht erkannt |
| Obst in Plastikverpackung | Erkennt evtl. die Verpackung |
| Mehrere Obstsorten im Bild | Nur die dominante wird erkannt |
| Gemälde eines Apfels | Funktioniert oft trotzdem! |

**Weiterführende Frage:** Wie würde man das Modell auf exotisches Obst erweitern?  
→ Stichwort: **Transfer Learning** / **Fine-Tuning**
